# FEVE Figures

Generates two sets of figures:
1. **Line plots** — FEVE vs. block index for frozen backbone and frozen+ft2blocks
2. **Violin plot** — Distribution of per-neuron FEVE for the best blocks (3 & 4, 0-indexed) across frozen / ft2blocks / ft3blocks

Figures are saved to `../figures/`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from utils import data, metrics
from utils.vit_encoding_model import build_vit_encoder, make_model_name, preprocess_images

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Configuration

**Fill in `CLUSTER_CHECKPOINT_BASE`** with the root path to your cluster checkpoint directory.  
Everything else is derived from the naming convention used during training.

In [ ]:
# ============================================================
# USER: set this to the root of your cluster checkpoints
# ============================================================
CLUSTER_CHECKPOINT_BASE = "/home/carsen/dm11_cluster/fengtongd/Desktop/github/minimodel-vit/"  # <-- FILL THIS IN

# Derived checkpoint directories (mirror the notebook layout on the cluster)
FROZEN_CKPT_DIR   = os.path.join(CLUSTER_CHECKPOINT_BASE, "notebooks", "checkpoints", "vit_frozen")
FINETUNE_CKPT_DIR = os.path.join(CLUSTER_CHECKPOINT_BASE, "notebooks", "checkpoints", "vit_finetune")

# Output directory for figures
FIGURES_DIR = "../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Data
DATA_PATH = "../data"
mouse_id  = 3  # FX10

HF_TOKEN = os.getenv("HF_TOKEN", "")  # HuggingFace token (needed to rebuild model architecture)

# Model architecture — must match training
MODEL_NAME = 'facebook/dinov3-vits16-pretrain-lvd1689m'
TOKEN_TYPE = 'patch'

# Block indices to sweep for the line plot
# Adjust to the exact set of blocks you trained on the cluster
FROZEN_BLOCKS  = list(range(12))   # blocks 0–11
FT2_BLOCKS     = list(range(12))   # blocks with ft2blocks checkpoints

# Best blocks for comparison figure (0-indexed transformer block index)
BEST_BLOCKS = [3, 4]

FEV_THRESHOLD = 0.15
BATCH_SIZE    = 64

## Load & preprocess data

In [ ]:
# Load raw images (no normalization yet)
img_raw = data.load_images(DATA_PATH, mouse_id,
                           file=data.img_file_name[mouse_id],
                           crop=False, normalize=False)
print('raw img shape:', img_raw.shape)

# Preprocess: 66×264 → (N, 3, 32, 64), ImageNet-normalized
img = preprocess_images(img_raw, batch_size=2000)
print('preprocessed:', img.shape)

In [ ]:
# Load neural data
fname = '%s_nat60k_%s.npz' % (data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(
    file_path=os.path.join(DATA_PATH, fname), mouse_id=mouse_id
)
n_stim, n_neurons = spks.shape
print('neurons:', n_neurons, '  test stimuli:', spks_rep_all.shape)

itrain, ival = data.split_train_val(istim_train, train_frac=0.9)
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)

ineur     = np.arange(n_neurons)
img_test  = torch.from_numpy(img[istim_test]).to(device)
print('img_test:', img_test.shape)

## Helper functions

In [ ]:
def frozen_ckpt_path(block):
    name = make_model_name(
        data.mouse_names[mouse_id], data.exp_date[mouse_id],
        MODEL_NAME, TOKEN_TYPE, [block]
    )
    return os.path.join(FROZEN_CKPT_DIR, name)


def finetune_ckpt_path(block, n_ft_blocks):
    base = make_model_name(
        data.mouse_names[mouse_id], data.exp_date[mouse_id],
        MODEL_NAME, TOKEN_TYPE, [block]
    ).replace('.pt', f'_ft{n_ft_blocks}blocks.pt')
    return os.path.join(FINETUNE_CKPT_DIR, base)


def load_model(ckpt_path, block):
    model = build_vit_encoder(
        n_neurons        = n_neurons,
        model_name       = MODEL_NAME,
        vit_input_size   = (64, 128),
        out_spatial_size = (4, 8),
        extract_layers   = [block],
        freeze_backbone  = True,
        poisson          = True,
        hf_token         = HF_TOKEN,
        device           = device,
    )
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model


def get_predictions(model):
    preds = []
    with torch.no_grad():
        for k in range(0, img_test.shape[0], BATCH_SIZE):
            preds.append(model(img_test[k:k + BATCH_SIZE]).cpu().numpy())
    return np.vstack(preds)


def compute_feve(predictions):
    """Returns (mean_feve_valid, fev_per_neuron, feve_per_neuron)."""
    fev, feve = metrics.feve(spks_rep_all, predictions)
    valid = fev > FEV_THRESHOLD
    return np.mean(feve[valid]), fev, feve


def evaluate_block(ckpt_path, block):
    """Load checkpoint, predict, compute FEVE. Returns None if ckpt missing."""
    if not os.path.exists(ckpt_path):
        print(f'  [skip] not found: {ckpt_path}')
        return None, None, None
    print(f'  loading block {block}: {os.path.basename(ckpt_path)}')
    model = load_model(ckpt_path, block)
    preds = get_predictions(model)
    del model
    torch.cuda.empty_cache()
    return compute_feve(preds)

## Sweep: frozen backbone (all blocks)

In [ ]:
# Results dictionaries: block → (mean_feve, fev_array, feve_array)
frozen_results = {}

print('Evaluating frozen backbone...')
for block in FROZEN_BLOCKS:
    mean_feve, fev, feve = evaluate_block(frozen_ckpt_path(block), block)
    if mean_feve is not None:
        frozen_results[block] = (mean_feve, fev, feve)
        print(f'  block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone. Available frozen blocks:', sorted(frozen_results.keys()))

## Sweep: fine-tuned 2 blocks unfrozen

In [ ]:
ft2_results = {}

print('Evaluating ft2blocks...')
for block in FT2_BLOCKS:
    mean_feve, fev, feve = evaluate_block(finetune_ckpt_path(block, n_ft_blocks=2), block)
    if mean_feve is not None:
        ft2_results[block] = (mean_feve, fev, feve)
        print(f'  block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone. Available ft2 blocks:', sorted(ft2_results.keys()))

## Evaluate ft1/ft3/ft4blocks for best blocks (3 & 4)

In [ ]:
ft1_results = {}
ft3_results = {}
ft4_results = {}

print('Evaluating ft1/ft3/ft4 blocks for best blocks...')
for n_ft, res_dict in [(1, ft1_results), (3, ft3_results), (4, ft4_results)]:
    print(f'\n  ft{n_ft}blocks:')
    for block in BEST_BLOCKS:
        mean_feve, fev, feve = evaluate_block(finetune_ckpt_path(block, n_ft_blocks=n_ft), block)
        if mean_feve is not None:
            res_dict[block] = (mean_feve, fev, feve)
            print(f'    block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone.')

## Figure style

In [ ]:
matplotlib.rcParams.update({
    'font.size':       11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':  1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'lines.linewidth': 2.0,
    'lines.markersize': 7,
})

COLORS = {
    'frozen': '#4878CF',   # blue
    'ft1':    '#F5A742',   # light orange
    'ft2':    '#E05C2A',   # dark orange
    'ft3':    '#2CA02C',   # green
    'ft4':    '#9467BD',   # purple
}

---
## Figure 1a — Frozen backbone: FEVE vs. block number

In [ ]:
frozen_blocks_sorted = sorted(frozen_results.keys())
frozen_feve_vals     = [frozen_results[b][0] for b in frozen_blocks_sorted]

fig, ax = plt.subplots(figsize=(6.5, 4))

ax.plot(
    frozen_blocks_sorted, frozen_feve_vals,
    color=COLORS['frozen'], marker='o', label='Frozen backbone'
)

ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('FEVE', fontsize=12)
ax.set_title(f'Frozen ViT-S/16 — FEVE across ViT blocks\n', fontsize=12)
ax.set_xticks(frozen_blocks_sorted)
ax.set_xticklabels([b+1 for b in frozen_blocks_sorted])  # 1-indexed for display
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.4)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig1a_frozen_feve_vs_block.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

---
## Figure 1b — Frozen + Fine-tuned (2 blocks): FEVE vs. block number

In [ ]:
# Shared x-axis: union of available blocks, sorted
ft2_blocks_sorted = sorted(ft2_results.keys())
ft2_feve_vals     = [ft2_results[b][0] for b in ft2_blocks_sorted]

fig, ax = plt.subplots(figsize=(6.5, 4))

ax.plot(
    frozen_blocks_sorted, frozen_feve_vals,
    color=COLORS['frozen'], marker='o', label='Frozen backbone'
)
ax.plot(
    ft2_blocks_sorted, ft2_feve_vals,
    color=COLORS['ft2'], marker='s', label='Fine-tuned (2 blocks)'
)

ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('FEVE', fontsize=12)
ax.set_title(f'Frozen vs. Fine-tuned ViT-S/16 — FEVE across ViT blocks\n', fontsize=12)

all_blocks = sorted(set(frozen_blocks_sorted) | set(ft2_blocks_sorted))
ax.set_xticks(all_blocks)
ax.set_xticklabels([b+1 for b in all_blocks])  # 1-indexed for display
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.6)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig1b_frozen_vs_ft2_feve_vs_block.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

---
## Figure 2 — Best blocks comparison: Frozen / FT-1 through FT-4 blocks

Violin plot showing the full distribution of per-neuron FEVE (valid neurons only)
for blocks 3 & 4 across all training conditions (frozen + 1–4 blocks unfrozen).

In [ ]:
# Build data arrays for violin plot
# Structure: conditions × blocks, each cell = array of per-neuron FEVE (valid only)

conditions = [
    ('Frozen',          frozen_results, COLORS['frozen']),
    ('FT (1 block)',    ft1_results,    COLORS['ft1']),
    ('FT (2 blocks)',   ft2_results,    COLORS['ft2']),
    ('FT (3 blocks)',   ft3_results,    COLORS['ft3']),
    ('FT (4 blocks)',   ft4_results,    COLORS['ft4']),
]

def valid_feve(results, block):
    """Per-neuron FEVE for valid neurons from a results dict."""
    if block not in results or results[block][0] is None:
        return None
    _, fev, feve = results[block]
    return feve[fev > FEV_THRESHOLD]


n_best   = len(BEST_BLOCKS)
n_cond   = len(conditions)
group_w  = 0.9          # total width per block group
viol_w   = group_w / n_cond * 0.85
offsets  = np.linspace(-group_w / 2 + viol_w / 2,
                        group_w / 2 - viol_w / 2, n_cond)

fig, ax = plt.subplots(figsize=(7, 4))

legend_handles = []

for c_idx, (label, res_dict, color) in enumerate(conditions):
    for b_idx, block in enumerate(BEST_BLOCKS):
        vals = valid_feve(res_dict, block)
        if vals is None:
            print(f'[warn] no data for block {block}, condition "{label}" — skipping')
            continue

        x_center = b_idx + offsets[c_idx]

        parts = ax.violinplot(
            vals, positions=[x_center], widths=viol_w,
            showmedians=True, showextrema=False
        )
        for pc in parts['bodies']:
            pc.set_facecolor(color)
            pc.set_alpha(0.6)
        parts['cmedians'].set_color(color)
        parts['cmedians'].set_linewidth(2)

        # Mean marker
        ax.scatter([x_center], [np.mean(vals)],
                   color=color, zorder=5, s=50, marker='D',
                   edgecolors='white', linewidths=0.8)

    # Legend proxy
    legend_handles.append(
        matplotlib.patches.Patch(facecolor=color, alpha=0.7, label=label)
    )

ax.set_xticks(range(n_best))
ax.set_xticklabels([f'Block {b+1}' for b in BEST_BLOCKS], fontsize=12)
ax.set_ylabel('FEVE (per neuron, valid only)', fontsize=12)
ax.set_title(f'FEVE distribution — Best blocks (frozen vs. fine-tuned)\n'
             f'diamonds = mean, lines = median', fontsize=11)
ax.set_xlim(-0.6, n_best - 0.4)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.legend(handles=legend_handles, frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 1.0)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig2_best_blocks_comparison.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## Summary table

In [ ]:
print(f"{'Block':>6}  {'Frozen':>10}  {'FT-1blk':>10}  {'FT-2blk':>10}  {'FT-3blk':>10}  {'FT-4blk':>10}  {'Best FT':>10}")
print('-' * 76)
for block in BEST_BLOCKS:
    frz  = frozen_results.get(block, (None,))[0]
    ft1  = ft1_results.get(block,    (None,))[0]
    ft2  = ft2_results.get(block,    (None,))[0]
    ft3  = ft3_results.get(block,    (None,))[0]
    ft4  = ft4_results.get(block,    (None,))[0]
    ft_vals = [v for v in [ft1, ft2, ft3, ft4] if v is not None]
    best_ft = max(ft_vals) if ft_vals else None
    fmt = lambda v: f'{v:.4f}' if v is not None else '   —   '
    print(f'{block+1:>6}  {fmt(frz):>10}  {fmt(ft1):>10}  {fmt(ft2):>10}  {fmt(ft3):>10}  {fmt(ft4):>10}  {fmt(best_ft):>10}')

---
## ViT-B/16 — Configuration & helper functions

In [ ]:
# ViT-B/16 model name  (adjust if your checkpoint uses a different key)
MODEL_NAME_B      = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
FROZEN_CKPT_DIR_B = os.path.join(CLUSTER_CHECKPOINT_BASE, "notebooks", "checkpoints", "vit_frozen")
FT_CKPT_DIR_B     = os.path.join(CLUSTER_CHECKPOINT_BASE, "notebooks", "checkpoints", "vit_finetune")

FROZEN_BLOCKS_B = list(range(12))
FT2_BLOCKS_B    = list(range(12))


def frozen_ckpt_path_b(block):
    name = make_model_name(
        data.mouse_names[mouse_id], data.exp_date[mouse_id],
        MODEL_NAME_B, TOKEN_TYPE, [block]
    )
    return os.path.join(FROZEN_CKPT_DIR_B, name)


def finetune_ckpt_path_b(block, n_ft_blocks):
    base = make_model_name(
        data.mouse_names[mouse_id], data.exp_date[mouse_id],
        MODEL_NAME_B, TOKEN_TYPE, [block]
    ).replace('.pt', f'_ft{n_ft_blocks}blocks.pt')
    return os.path.join(FT_CKPT_DIR_B, base)


def load_model_b(ckpt_path, block):
    model = build_vit_encoder(
        n_neurons        = n_neurons,
        model_name       = MODEL_NAME_B,
        vit_input_size   = (64, 128),
        out_spatial_size = (4, 8),
        extract_layers   = [block],
        freeze_backbone  = True,
        poisson          = True,
        hf_token         = HF_TOKEN,
        device           = device,
    )
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model


def evaluate_block_b(ckpt_path, block):
    """Load ViT-B/16 checkpoint, predict, compute FEVE. Returns None if ckpt missing."""
    if not os.path.exists(ckpt_path):
        print(f'  [skip] not found: {ckpt_path}')
        return None, None, None
    print(f'  loading block {block}: {os.path.basename(ckpt_path)}')
    model = load_model_b(ckpt_path, block)
    preds = get_predictions(model)
    del model
    torch.cuda.empty_cache()
    return compute_feve(preds)


print('Example ViT-B/16 frozen ckpt name:', os.path.basename(frozen_ckpt_path_b(0)))


## Sweep: ViT-B/16 frozen backbone (all blocks)

In [ ]:
frozen_results_b = {}

print('Evaluating ViT-B/16 frozen backbone...')
for block in FROZEN_BLOCKS_B:
    mean_feve, fev, feve = evaluate_block_b(frozen_ckpt_path_b(block), block)
    if mean_feve is not None:
        frozen_results_b[block] = (mean_feve, fev, feve)
        print(f'  block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone. Available ViT-B/16 frozen blocks:', sorted(frozen_results_b.keys()))


## Sweep: ViT-B/16 fine-tuned (2 blocks unfrozen)

In [ ]:
ft2_results_b = {}

print('Evaluating ViT-B/16 ft2blocks...')
for block in FT2_BLOCKS_B:
    mean_feve, fev, feve = evaluate_block_b(finetune_ckpt_path_b(block, n_ft_blocks=2), block)
    if mean_feve is not None:
        ft2_results_b[block] = (mean_feve, fev, feve)
        print(f'  block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone. Available ViT-B/16 ft2 blocks:', sorted(ft2_results_b.keys()))


## Evaluate ft1/ft3/ft4blocks for ViT-B/16 best blocks (3 & 4)

In [ ]:
ft1_results_b = {}
ft3_results_b = {}
ft4_results_b = {}

print('Evaluating ViT-B/16 ft1/ft3/ft4 blocks for best blocks...')
for n_ft, res_dict in [(1, ft1_results_b), (3, ft3_results_b), (4, ft4_results_b)]:
    print(f'\n  ft{n_ft}blocks:')
    for block in BEST_BLOCKS:
        mean_feve, fev, feve = evaluate_block_b(finetune_ckpt_path_b(block, n_ft_blocks=n_ft), block)
        if mean_feve is not None:
            res_dict[block] = (mean_feve, fev, feve)
            print(f'    block {block:2d}  FEVE = {mean_feve:.4f}')

print('\nDone.')

---
## Figure 2b — ViT-S/16 vs ViT-B/16: best blocks, frozen + fine-tuned

Side-by-side violin plots showing per-neuron FEVE distribution for blocks 3 & 4
across all training conditions, comparing both model sizes.

In [ ]:
conditions_s = [
    ('Frozen',        frozen_results,   COLORS['frozen']),
    ('FT (1 block)',  ft1_results,      COLORS['ft1']),
    ('FT (2 blocks)', ft2_results,      COLORS['ft2']),
    ('FT (3 blocks)', ft3_results,      COLORS['ft3']),
    ('FT (4 blocks)', ft4_results,      COLORS['ft4']),
]
conditions_b = [
    ('Frozen',        frozen_results_b, COLORS['frozen']),
    ('FT (1 block)',  ft1_results_b,    COLORS['ft1']),
    ('FT (2 blocks)', ft2_results_b,    COLORS['ft2']),
    ('FT (3 blocks)', ft3_results_b,    COLORS['ft3']),
    ('FT (4 blocks)', ft4_results_b,    COLORS['ft4']),
]

n_best  = len(BEST_BLOCKS)
n_cond  = len(conditions_s)
group_w = 0.9
viol_w  = group_w / n_cond * 0.85
offsets = np.linspace(-group_w / 2 + viol_w / 2,
                       group_w / 2 - viol_w / 2, n_cond)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, conditions, title in zip(
    axes,
    [conditions_s, conditions_b],
    ['ViT-S/16', 'ViT-B/16']
):
    legend_handles = []
    for c_idx, (label, res_dict, color) in enumerate(conditions):
        for b_idx, block in enumerate(BEST_BLOCKS):
            vals = valid_feve(res_dict, block)
            if vals is None:
                continue
            x_center = b_idx + offsets[c_idx]
            parts = ax.violinplot(vals, positions=[x_center], widths=viol_w,
                                  showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.6)
            parts['cmedians'].set_color(color)
            parts['cmedians'].set_linewidth(2)
            ax.scatter([x_center], [np.mean(vals)], color=color, zorder=5,
                       s=50, marker='D', edgecolors='white', linewidths=0.8)
        legend_handles.append(
            matplotlib.patches.Patch(facecolor=color, alpha=0.7, label=label)
        )

    ax.set_xticks(range(n_best))
    ax.set_xticklabels([f'Block {b+1}' for b in BEST_BLOCKS], fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
    ax.legend(handles=legend_handles, frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlim(-0.6, n_best - 0.4)
    ax.set_ylim(-0.1, 1.0)

axes[0].set_ylabel('FEVE', fontsize=12)
fig.suptitle('FEVE distribution — Best blocks: ViT-S/16 vs ViT-B/16\ndiamonds = mean, lines = median',
             fontsize=11)
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig2b_vits_vs_vitb_best_blocks.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

In [ ]:
# summary table of ViT-S/16 vs ViT-B/16 best blocks
rows = [
    ('ViT-S/16', frozen_results,   ft1_results,   ft2_results,   ft3_results,   ft4_results),
    ('ViT-B/16', frozen_results_b, ft1_results_b, ft2_results_b, ft3_results_b, ft4_results_b),
]

print(f"{'Model':>10}  {'Block':>6}  {'Frozen':>10}  {'FT-1blk':>10}  {'FT-2blk':>10}  {'FT-3blk':>10}  {'FT-4blk':>10}  {'Best FT':>10}")
print('-' * 90)
for model_label, frz_res, ft1_res, ft2_res, ft3_res, ft4_res in rows:
    for block in BEST_BLOCKS:
        frz = frz_res.get(block, (None,))[0]
        ft1 = ft1_res.get(block, (None,))[0]
        ft2 = ft2_res.get(block, (None,))[0]
        ft3 = ft3_res.get(block, (None,))[0]
        ft4 = ft4_res.get(block, (None,))[0]
        ft_vals = [v for v in [ft1, ft2, ft3, ft4] if v is not None]
        best_ft = max(ft_vals) if ft_vals else None
        fmt = lambda v: f'{v:.4f}' if v is not None else '   —   '
        print(f'{model_label:>10}  {block+1:>6}  {fmt(frz):>10}  {fmt(ft1):>10}  {fmt(ft2):>10}  {fmt(ft3):>10}  {fmt(ft4):>10}  {fmt(best_ft):>10}')
    print()

---
## Figure 3 — ViT-S/16 vs ViT-B/16: frozen backbone, FEVE vs block

In [ ]:
# --- collect S and B frozen series ---
s_blocks = sorted(frozen_results.keys())
s_feve   = [frozen_results[b][0] for b in s_blocks]

b_blocks = sorted(frozen_results_b.keys())
b_feve   = [frozen_results_b[b][0] for b in b_blocks]

fig, ax = plt.subplots(figsize=(6.5, 4))

ax.plot(s_blocks, s_feve,
        color='#4878CF', marker='o', label='ViT-S/16 frozen')
ax.plot(b_blocks, b_feve,
        color='#D62728', marker='s', label='ViT-B/16 frozen')

all_blocks = sorted(set(s_blocks) | set(b_blocks))
ax.set_xticks(all_blocks)
ax.set_xticklabels([b+1 for b in all_blocks])  # 1-indexed for display
ax.set_xlabel('Block index', fontsize=12)
ax.set_ylabel('FEVE', fontsize=12)
ax.set_title(f'ViT-S/16 vs ViT-B/16 — Frozen backbone\n',
             fontsize=12)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.45)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig3_vits_vs_vitb_frozen_feve_vs_block.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()


---
## Figure 4 — ViT-S/16 vs ViT-B/16: frozen + fine-tuned, FEVE vs block

In [ ]:
# --- collect ft2 series ---
s_ft2_blocks = sorted(ft2_results.keys())
s_ft2_feve   = [ft2_results[b][0] for b in s_ft2_blocks]

b_ft2_blocks = sorted(ft2_results_b.keys())
b_ft2_feve   = [ft2_results_b[b][0] for b in b_ft2_blocks]

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(s_blocks,     s_feve,     color='#4878CF', marker='o',
        linestyle='-',  label='ViT-S/16 frozen')
ax.plot(s_ft2_blocks, s_ft2_feve, color='#4878CF', marker='o',
        linestyle='--', label='ViT-S/16 ft2blocks')

ax.plot(b_blocks,     b_feve,     color='#D62728', marker='s',
        linestyle='-',  label='ViT-B/16 frozen')
ax.plot(b_ft2_blocks, b_ft2_feve, color='#D62728', marker='s',
        linestyle='--', label='ViT-B/16 ft2blocks')

all_blocks = sorted(set(s_blocks) | set(s_ft2_blocks) | set(b_blocks) | set(b_ft2_blocks))
ax.set_xticks(all_blocks)
ax.set_xticklabels([b+1 for b in all_blocks])  # 1-indexed for display
ax.set_xlabel('Block index', fontsize=12)
ax.set_ylabel('FEVE', fontsize=12)
ax.set_title(f'ViT-S/16 vs ViT-B/16 — Frozen & Fine-tuned\n',
             fontsize=12)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.legend(frameon=False, fontsize=9, ncol=2, handlelength=3)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.6)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig4_vits_vs_vitb_frozen_and_ft_feve_vs_block.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)
plt.show()


---
## Figure 5 — Best block performance: ViT-S/16 vs ViT-B/16

Bar chart comparing mean FEVE at the best block for each model, for both frozen and ft2blocks conditions.

In [ ]:
def best_block_mean_feve(results):
    """Return (best_block_index, best_mean_feve) from a results dict."""
    if not results:
        return None, None
    best_b = max(results, key=lambda b: results[b][0])
    return best_b, results[best_b][0]


def best_block_feve_array(results):
    """Return per-neuron FEVE array for the best block (valid neurons only)."""
    if not results:
        return None, None
    best_b = max(results, key=lambda b: results[b][0])
    _, fev, feve = results[best_b]
    return best_b, feve[fev > FEV_THRESHOLD]


models = [
    ('ViT-S/16\nfrozen',   frozen_results,   '#4878CF'),
    ('ViT-S/16\nft4blks',  ft4_results,      COLORS['ft4']),
    ('ViT-B/16\nfrozen',   frozen_results_b, '#D62728'),
    ('ViT-B/16\nft4blks',  ft4_results_b,    '#E88080'),
]

labels   = []
feve_arr = []
colors   = []
best_info = []

for label, res_dict, color in models:
    best_b, arr = best_block_feve_array(res_dict)
    labels.append(label)
    feve_arr.append(arr)
    colors.append(color)
    mean_v = arr.mean() if arr is not None else float('nan')
    best_info.append((label.replace('\n', ' '), best_b, mean_v))

# --- figure ---
fig, ax = plt.subplots(figsize=(6, 4))

positions = range(len(labels))
for pos, (arr, color, label) in enumerate(zip(feve_arr, colors, labels)):
    if arr is None:
        print(f'[warn] no data for {label}')
        continue
    parts = ax.violinplot(arr, positions=[pos], widths=0.65,
                          showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.65)
    parts['cmedians'].set_color(color)
    parts['cmedians'].set_linewidth(2)
    ax.scatter([pos], [arr.mean()], color=color, zorder=5, s=60,
               marker='D', edgecolors='white', linewidths=0.8)

ax.set_xticks(list(positions))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('FEVE', fontsize=12)
ax.set_title(f'Best-block FEVE — ViT-S/16 vs ViT-B/16\ndiamonds = mean, lines = median',
             fontsize=11)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 1.0)
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'fig5_best_block_vits_vs_vitb.pdf')
fig.savefig(save_path, dpi=200, bbox_inches='tight')
fig.savefig(save_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print('Saved:', save_path)

print('\nBest block summary:')
print(f'  {"Model":<22}  {"Best block":>10}  {"Mean FEVE":>10}')
print('  ' + '-' * 46)
for label, bb, mf in best_info:
    bb_str = str(bb+1) if bb is not None else '-'
    mf_str = f'{mf:.4f}' if bb is not None else '-'
    print(f'  {label:<22}  {bb_str:>10}  {mf_str:>10}')

plt.show()